# ETL - Limpeza e Preparação dos Dados
Carregamento, limpeza, validação e exportação dos datasets para o banco SQLite.

In [1]:
import pandas as pd
import json
import sqlite3
import re

## 1. Carregamento dos dados brutos

In [2]:
df_vendas   = pd.read_csv('/home/vinic/Documentos/Projetos/LH_Nautical/datasets/vendas_2023_2024.csv')
df_produtos = pd.read_csv('/home/vinic/Documentos/Projetos/LH_Nautical/datasets/produtos_raw.csv')

with open('/home/vinic/Documentos/Projetos/LH_Nautical/datasets/clientes_crm.json') as f:
    df_clientes = pd.DataFrame(json.load(f))

with open('/home/vinic/Documentos/Projetos/LH_Nautical/datasets/custos_importacao.json') as f:
    custos_raw = json.load(f)

## 2. Diagnóstico inicial

In [3]:
for nome, df in [('vendas', df_vendas), ('produtos', df_produtos), ('clientes', df_clientes)]:
    print(f"\n{'='*30} {nome.upper()} {'='*30}")
    print(df.info())
    print(df.head(3))


============================== VENDAS ==============================
<class 'pandas.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          9895 non-null   int64  
 1   id_client   9895 non-null   int64  
 2   id_product  9895 non-null   int64  
 3   qtd         9895 non-null   int64  
 4   total       9895 non-null   float64
 5   sale_date   9895 non-null   str    
dtypes: float64(1), int64(4), str(1)
memory usage: 464.0 KB
None
   id  id_client  id_product  qtd    total   sale_date
0   0         42         105   11   3405.0  2023-09-10
1   1          3         136    9  16873.9  15-09-2024
2   2         25         139    7   9475.3  2024-08-13

============================== PRODUTOS ==============================
<class 'pandas.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------  

## 3. Limpeza — Clientes
- Corrigir emails com `#` no lugar de `@`
- Separar `location` em colunas `city` e `state`

In [4]:
# Corrigindo os emails
mask_invalidos = ~df_clientes['email'].str.contains('@', regex=False)
print("Emails inválidos:", df_clientes[mask_invalidos]['email'].tolist())
df_clientes['email'] = df_clientes['email'].str.replace('#', '@', regex=False)

# Separar cidade e estado
def extrair_estado(loc):
    match = re.search(r'\b([A-Z]{2})\b', loc)
    return match.group(1) if match else None

def extrair_cidade(loc):
    cidade = re.sub(r'\b[A-Z]{2}\b', '', loc)
    return cidade.replace(',', '').strip(' ,-()')

df_clientes['state'] = df_clientes['location'].apply(extrair_estado)
df_clientes['city']  = df_clientes['location'].apply(extrair_cidade)
df_clientes = df_clientes.drop(columns=['location'])

df_clientes.head(5)

Emails inválidos: ['farias.teixeira.daniel.ribeiro#gmail.com', 'thiago.moreira#gmail.com', 'pedro.freitas#icloud.com', 'torres.barros.rocha.bianca.siqueira#aol.com', 'pimentel.alves.luiz#outlook.com', 'lucas.lopes.guedes.cunha#tutanota.com', 'paiva.débora#gmx.com', 'rafael.pereira.barros#zoho.com', 'martins.guimarães.carlos#hotmail.com', 'lopes.alves.pacheco.rocha.carla#yahoo.com', 'rocha.adriana.guedes.borges.alves#protonmail.com', 'peixoto.soares.rocha.oliveira.sandra#yahoo.com', 'borges.vieira.daniela.mendonça.farias#protonmail.com', 'rodrigues.mendonça.bruna#aol.com', 'peixoto.carla.araújo.silva.azevedo#outlook.com', 'gomes.araújo.borges.luiz#hotmail.com', 'jéssica.martins.leite.figueiredo#tutanota.com', 'guedes.cunha.costa.diego#outlook.com', 'nunes.cunha.victor.vieira#protonmail.com', 'martins.priscila#tutanota.com', 'nunes.borges.teixeira.pimentel.jéssica#yahoo.com', 'mateus.antunes#gmail.com', 'pacheco.lucas#aol.com', 'costa.correia.ribeiro.santos.pedro#yahoo.com', 'márcia.figu

,full_name,code,email,state,city
0,Femininos Oliveira Antunes,1,femininos.oliveira.antunes@icloud.com,BA,Aratu (Candeias
1,Fernanda Azevedo Soares Nunes Vieira,2,nunes.fernanda.soares.azevedo.vieira@outlook.com,PE,Recife
2,Daniel Farias Ribeiro Teixeira,3,farias.teixeira.daniel.ribeiro@gmail.com,RS,Rio Grande
3,Thiago Moreira,4,thiago.moreira@gmail.com,AC,Rio Branco
4,Pedro Freitas,5,pedro.freitas@icloud.com,PA,Santarém Novo


## 4. Limpeza — Vendas
- Padronizar datas

In [5]:
def parse_data(d):
    for fmt in ('%Y-%m-%d', '%d-%m-%Y'):
        try:
            return pd.to_datetime(d, format=fmt)
        except:
            pass
    return pd.NaT

df_vendas['sale_date'] = df_vendas['sale_date'].apply(parse_data)
print("Datas inválidas:", df_vendas['sale_date'].isna().sum())
df_vendas.head(5)

Datas inválidas: 0


,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,2024-09-15
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


## 5. Limpeza — Produtos
- Converter `price` de texto (`R$ x`) para float
- Normalizar `actual_category`

In [6]:
# Preço
df_produtos['price'] = (
    df_produtos['price']
    .str.replace('R$', '', regex=False)
    .str.strip()
    .astype(float)
)

# Categorias
def normalizar_categoria(cat):
    cat = cat.lower()
    cat = re.sub(r'\s+', '', cat)
    cat = cat.replace('ô', 'o').replace('ã', 'a').replace('ç', 'c')
    if 'eletr' in cat:   return 'ELETRONICOS'
    elif 'prop' in cat or 'puls' in cat: return 'PROPULSAO'
    elif 'ancor' in cat or 'encor' in cat: return 'ANCORAGEM'
    return 'OUTROS'

df_produtos['category'] = df_produtos['actual_category'].apply(normalizar_categoria)
df_produtos = df_produtos.drop(columns=['actual_category'])

print("Categorias:", df_produtos['category'].unique())
df_produtos.head(5)

Categorias: <StringArray>
['ELETRONICOS', 'PROPULSAO', 'ANCORAGEM']
Length: 3, dtype: str


,name,price,code,category
0,Transponder AIS Maré Magnum,33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,9024.19,3,ELETRONICOS
3,Rádio AIS Hydro Tidal Zen,3381.88,4,ELETRONICOS
4,Piloto Automático Furuno Storm,23669.01,5,ELETRONICOS


## 6. Limpeza — Custos

In [7]:
registros = []
for item in custos_raw:
    for h in item['historic_data']:
        registros.append({
            'product_id':   item['product_id'],
            'product_name': item['product_name'],
            'category':     item['category'],
            'start_date':   pd.to_datetime(h['start_date'], dayfirst=True),
            'usd_price':    h['usd_price']
        })

df_custos = pd.DataFrame(registros)
print(f"Custos: {df_custos.shape}")
df_custos.head(5)

Custos: (1260, 5)


,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,2016-08-10,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,2018-06-15,8778.36
2,1,Transponder AIS Maré Magnum,eletrônicos,2018-09-25,8023.87
3,1,Transponder AIS Maré Magnum,eletrônicos,2019-03-19,8772.78
4,1,Transponder AIS Maré Magnum,eletrônicos,2020-01-17,7918.18


## 7. Validação final

In [8]:
print("VENDAS")
print(f"  Shape: {df_vendas.shape} | Nulls: {df_vendas.isnull().sum().sum()} | sale_date: {df_vendas['sale_date'].dtype}")

print("\nPRODUTOS")
print(f"  Shape: {df_produtos.shape} | Nulls: {df_produtos.isnull().sum().sum()} | Categorias: {df_produtos['category'].unique()}")
print(df_produtos[['name', 'price']].head(3))
print(df_produtos['price'].describe())

print("\nCLIENTES")
print(f"  Shape: {df_clientes.shape} | Nulls: {df_clientes.isnull().sum().sum()}")

print("\nCUSTOS")
print(f"  Shape: {df_custos.shape} | Nulls: {df_custos.isnull().sum().sum()}")

# Integridade referencial
orfaos_prod = set(df_vendas['id_product'].unique()) - set(df_produtos['code'].unique())
orfaos_cli  = set(df_vendas['id_client'].unique())  - set(df_clientes['code'].unique())
print(f"\nProdutos órfãos: {orfaos_prod}")
print(f"Clientes órfãos: {orfaos_cli}")

VENDAS
  Shape: (9895, 6) | Nulls: 0 | sale_date: datetime64[us]

PRODUTOS
  Shape: (157, 4) | Nulls: 0 | Categorias: <StringArray>
['ELETRONICOS', 'PROPULSAO', 'ANCORAGEM']
Length: 3, dtype: str
                           name     price
0   Transponder AIS Maré Magnum  33122.52
1     Transponder Furuno Marlin  13998.15
2  Radar Furuno Pulse Leviathan   9024.19
count       157.000000
mean      35194.622484
std       42183.183409
min         309.540000
25%        3769.930000
50%       13704.100000
75%       51634.040000
max      148198.230000
Name: price, dtype: float64

CLIENTES
  Shape: (49, 5) | Nulls: 0

CUSTOS
  Shape: (1260, 5) | Nulls: 0

Produtos órfãos: set()
Clientes órfãos: set()


## 8. Exportar para SQLite

In [9]:
DB_PATH = '/home/vinic/Documentos/Projetos/LH_Nautical/datasets/nautical.db'
conn = sqlite3.connect(DB_PATH)

df_vendas.to_sql('vendas',    conn, if_exists='replace', index=False)
df_produtos.to_sql('produtos', conn, if_exists='replace', index=False)
df_clientes.to_sql('clientes', conn, if_exists='replace', index=False)
df_custos.to_sql('custos',    conn, if_exists='replace', index=False)

conn.close()
print("Banco criado:", DB_PATH)

Banco criado: /home/vinic/Documentos/Projetos/LH_Nautical/datasets/nautical.db


## 9. Verificação do banco

In [10]:
conn = sqlite3.connect(DB_PATH)
for tabela in ['vendas', 'produtos', 'clientes', 'custos']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {tabela}", conn).iloc[0,0]
    print(f"  {tabela}: {count} registros")
conn.close()

  vendas: 9895 registros
  produtos: 157 registros
  clientes: 49 registros
  custos: 1260 registros
